# Uncertain decision problem {#sec-uncertain-decision-problem}

Here we describe the model environment of the uncertain decision probles and provide two Python classes, one for the single-agent environment, the other for the two-agent environment. The interface to the learning dynamics is models according to the first author's Python package [pyCRLD](https://www.github.com/barfusslab/pyCRLD). Note that we reimplement the relevant parts of pyCRLD in @sec-learning-dynamics to keep dependencies minimal.

In [1]:
 #| default_exp UncertainDecisionProblem

In [2]:
 #| export
import numpy as np
import itertools as it
from scipy.stats import norm
from fastcore.utils import patch
import matplotlib.pyplot as plt
np.set_printoptions(legacy='1.25')

The environment of the multiagent-environment interface is composed of two parts: the natural environment and the social environment.

The natural environment consists of two states, A and B. In each episode, the natural environment is in state A with probability $p_A$ and in state B with probability $1-p_A$. 

Agent can choose between two actions, A and B. Depending on the nature state, agents receive a positive reward if they choose the action corresponding to the state of the natural environment and -1 otherwise.

## Observation model

However, the agents cannot directly observe the true state of nature. They only receive a noisy signal which can take one of four values: *A, a, b, B*. Here, 

* *A* denotes nature to be *very likely* in state A, 
* *a* denotes nature to be *likely* in state A, 
* *b* denotes nature to be *likely* in state B, and
* *B* denotes nature to be *very likely* in state B.


The probability of receiving each signal $o$, $O^{so}$, depends on the true state of nature $s$ and the noiselevel $\sigma$.
We model the probability of receiving each signal with a normal distribution $\mathcal N_{\mu,\sigma}$ centered around the true state of nature (suppose $\mu=0$) and the other nature state one unit length apart (at 1). The noise levels $\sigma$ corresponds to the standard deviation of the normal distribution. Agents make observation 

* *A* with probability $\int\limits_{-\infty}^{-d} \mathcal N_{0,\sigma}(x)\, dx$,
* *a* with probability $\int\limits_{-d}^{0.5} \mathcal N_{0,\sigma}(x)\, dx$,
* *b* with probability $\int\limits_{0.5}^{1+d} \mathcal N_{0,\sigma}(x)\, dx$, and
* *B* with probability $\int\limits_{1+d}^{\infty} \mathcal N_{0,\sigma}(x)\, dx$, 

where $d$ is the distance from the center of nature's true state at which the agent believes to observe the state *very* likely.




The following Python function illustrates this observation model.

In [3]:
 #| export
def plot_observation_function(noise_level = 1, 
                              distance = 0.5,
                              x_extend = 3):
    x = np.linspace(-x_extend, x_extend, 1000)
    plt.plot(x, norm.pdf(x, loc=0, scale=noise_level), 'k-')

    x1 = np.linspace(-x_extend, -distance, 1000)
    plt.fill_between(x1, norm.pdf(x1, loc=0, scale=noise_level),
                     color='darkgreen', alpha=0.5)

    x2 = np.linspace(-distance, 0.5, 1000)
    plt.fill_between(x2, norm.pdf(x2, loc=0, scale=noise_level),
                     color='lightgreen', alpha=0.5)

    x3 = np.linspace(0.5, 1+distance, 1000)
    plt.fill_between(x3, norm.pdf(x3, loc=0, scale=noise_level),
                     color='violet', alpha=0.5)

    x4 = np.linspace(1+distance, x_extend, 1000)
    plt.fill_between(x4, norm.pdf(x4, loc=0, scale=noise_level),
                     color='purple', alpha=0.5)

    # Get current xticks and labels
    locations, labels = plt.xticks()

    # Create new labels
    new_labels = ['' for _ in labels]
    for i, loc in enumerate(locations):
        if abs(loc - 0) < 1e-2: new_labels[i] = 'A'
        elif abs(loc - 1) < 1e-2: new_labels[i] = 'B'

    # Set new labels
    plt.xticks(locations, new_labels)
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.xlim(-x_extend, x_extend)

Assuming *A* is the true state of nature, the shaded areas indicate the probability of observing each signal. The noise level $\sigma=1.0$ and $d=0.5$.

In [4]:
#| fig-cap: Illustration of the observational uncertainty function
plot_observation_function(noise_level=1, distance=0.5)

<Figure size 2700x2100 with 1 Axes>

We compute these observation probabilities as follows.

In [5]:
 #|export
def probabilities(noiselevel, likelydistance):
    """
    Returns observation probabilities. 
    """
    pVeryLikelyTrue = norm.cdf(-likelydistance, 0,  noiselevel)        # A
    pLikelyTrue = norm.cdf(0.5, 0, noiselevel) - pVeryLikelyTrue       # a  
    pLikelyWrong = norm.cdf(1+likelydistance, 0, noiselevel)\
        - norm.cdf(0.5, 0, noiselevel)                                 # b
    pVeryLikelyWrong = 1 -  norm.cdf(1+likelydistance, 0, noiselevel)  # B
    
    return [pVeryLikelyTrue, pLikelyTrue, pLikelyWrong, pVeryLikelyWrong]

In [6]:
probabilities(noiselevel=2.7, likelydistance=0.5)

[0.42654189431596573,
 0.1469162113680686,
 0.13728453356199366,
 0.289257360753972]

We observe how the observation probabilities change as we change the noise level, $\sigma$ and the distance $d$.

In [7]:
 #|export
def plot_observation_probabilities(max_noiselevel, likelydistance=0.5):
    noiselevels = np.linspace(0.01, max_noiselevel, 200)
    probs = np.array([probabilities(sig, likelydistance) for sig in noiselevels])
    plt.plot(noiselevels, probs[:,0], '.-', label= 'A', color = 'darkgreen')
    plt.plot(noiselevels, probs[:,1], '.-', label= 'a', color = 'lightgreen')
    plt.plot(noiselevels, probs[:,2], '.-', label= 'b', color = 'violet')
    plt.plot(noiselevels, probs[:,3], '.-', label= 'B', color = 'purple')
    
    plt.plot(noiselevels, probs[:,0]+probs[:,1], '--', label= 'A+a', color = 'green')
    plt.plot(noiselevels, probs[:,2]+probs[:,3], '--', label= 'B+b', color = 'pink')

    plt.xlabel('Observational uncertainty $\\sigma$')
    plt.ylabel('Observation probabilities')
    plt.legend()

In [8]:
#| fig-cap: Observation probabilities as a function of observational uncertainty
plot_observation_probabilities(max_noiselevel=4, likelydistance=0.5)

<Figure size 2700x2100 with 1 Axes>

## Environment classes

We first define an environmental base class defining some common attributes and implementing checks for internal consistency.

In [9]:
 #| export
class ebase(object):
    """Base environment. All environments should inherit from this one."""
    
    def __init__(self):
               
        self.T = self.TransitionTensor()
        self.R = self.RewardTensor()
        self.O = self.ObservationTensor()
                
        self.Aset = self.actions()
        self.Sset = self.states() 
        self.Oset = self.observations()

        # CHECKS
        R, T, O = self.R, self.T, self.O
        
        # number of agents
        N = R.shape[0]  
        assert O.shape[0] == N, "Inconsistent number of agents"
        assert len(T.shape[1:-1]) == N, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == N, "Inconsistent number of agents"
        
        # number of actions for each agent        
        M = T.shape[1] 
        assert np.allclose(T.shape[1:-1], M), 'Inconsistent number of actions'
        assert np.allclose(R.shape[2:-1], M), 'Inconsistent number of actions'
        assert np.all(list(map(len, self.Aset)) == np.array(M).repeat(N)),\
            'Inconsistent number of actions'
            
        # number of states
        Z = T.shape[0] 
        assert T.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[1] == Z, 'Inconsistent number of states'
        assert O.shape[1] == Z, 'Inconsistent number of states'
        assert len(self.Sset) == Z, 'Inconsistent number of states'

        # number of observations
        Q = O.shape[-1]
        assert np.all(list(map(len, self.Oset)) == np.array(Q).repeat(N)),\
            'Inconsistent number of observations'
        
        assert np.allclose(T.sum(-1), 1), 'Transition model wrong'
        assert np.allclose(O.sum(-1), 1), 'Observation model wrong'


## Single-agent environment

Then, we summarize the observaton logic from above into a single-agent environment Python class `SingleAgentUncertainDecisionProblem` which can be used with the learning dynamics (as implement in the @sec-learning-dynamics).

In [10]:
 #| export
class SingleAgentUncertainDecisionProblem(ebase):

    def __init__(self, 
                 noiselevel: float, # standard deviation of natural uncertainty
                 probabilityA: float = 0.5, # probability of nature to be in A
                 likelydistance: float = 1): # distance when agent's believe become *very* likely
        self.env_A = probabilityA
        self.noise = noiselevel
        self.likelydistance = likelydistance
        self.probs = probabilities(noiselevel, likelydistance)

        self.N = 1
        self.M = len(self.actions()[0])
        self.Z = len(self.states())
        self.Q = len(self.observations()[0])

        # inital states
        super().__init__()
        # self.state = np.random.choice(np.where(self.F==0)[0])

In the single-agent environment, the environmental states consist only of nature states. 

In [11]:
 #| export
@patch
def states(self:SingleAgentUncertainDecisionProblem): 
    return ["A", "B"]

The agent's action set,

In [12]:
 #| export
@patch
def actions(self:SingleAgentUncertainDecisionProblem): 
    return [['A', 'B']]    

and its observation set,

In [13]:
 #| export
@patch
def observations(self:SingleAgentUncertainDecisionProblem): 
    return [['A', 'a', 'b', 'B']]

The transition tensor is very simple. It only depends on the base probability $p_A$.

In [14]:
 #| export
@patch
def TransitionTensor(self:SingleAgentUncertainDecisionProblem):
    dim = np.concatenate(([self.Z],
                          [self.M for _ in range(self.N)],
                          [self.Z]))
    Tsas = np.ones(dim) * (-1)

    for index, _ in np.ndenumerate(Tsas):
        Tsas[index] = self._transition_probability(index[0],
                                                    index[1:-1],
                                                    index[-1])
    return Tsas

@patch
def _transition_probability(self:SingleAgentUncertainDecisionProblem, 
                            s, jA, s_):
        env_ = self.states()[s_]
        if env_ == 'A':
            return self.env_A
        else:
            return 1-self.env_A

The reward tensor yields, 

In [15]:
 #| export
@patch
def RewardTensor(self:SingleAgentUncertainDecisionProblem):
    """Get the Reward Tensor R[i,s,a1,...,aN,s']."""
    dim = np.concatenate(([self.N],
                          [self.Z],
                          [self.M for _ in range(self.N)],
                          [self.Z]))
    Risas = np.zeros(dim)

    for index, _ in np.ndenumerate(Risas):
        Risas[index] = self._reward(index[0], index[1], index[2:-1], index[-1])
    return Risas
    
@patch
def _reward(self:SingleAgentUncertainDecisionProblem, i, s, jA, s_):
        # translating states and actions into coordinates
        env = self.states()[s]
        act = self.actions()[i][jA[0]]
            
        reward = 0.0     
        if env == act: reward = 1.0
        elif env != act: reward = -1.0
        return reward

The observation tensor is implemented according to the observation model described above,

In [16]:
 #| export
@patch
def ObservationTensor(self:SingleAgentUncertainDecisionProblem):
    """Get the Observation Tensor O[i,s,o]."""
    Oiso = np.ones((self.N, self.Z, self.Q))
    
    for index, _ in np.ndenumerate(Oiso):
        Oiso[index] = self._observation(index[0], index[1], index[2])
    return Oiso
    
@patch
def _observation(self:SingleAgentUncertainDecisionProblem, i, s, o):
    # translating states and actions into coordinates    
    env = self.states()[s]
    eobs = self.observations()[i][o]

    # observations probability
    p = 0.0
                    
    if eobs.upper() == env and eobs.isupper():
        p = self.probs[0]
    if eobs.upper() == env and not eobs.isupper():
        p = self.probs[1]
    if eobs.upper() != env and not eobs.isupper():
        p = self.probs[2]
    if eobs.upper() != env and eobs.isupper():
        p = self.probs[3]
        
    return p

Last, the environment gets an `id` method return a string identifier of the environment to be used in saving and retrieving results data.

In [17]:
 #| export
@patch
def id(self:SingleAgentUncertainDecisionProblem):
    """
    Returns id string of environment
    """
    id = f"{self.__class__.__name__}_"+\
        f"{self.noise}_{self.env_A}_{self.likelydistance}"
    return id

We execute some basic tests of the environment.

In [18]:
#| output: asis
env = SingleAgentUncertainDecisionProblem(noiselevel=1.0,
                                          probabilityA=0.5,
                                          likelydistance=0.5)
print("\\begin{OutputCode}")  #| hide_line
print(env.id())
print()
print("State set:", env.Sset)
print()
print("Observation set:", env.Oset[0])
print()
print("Action set:", env.Aset[0])
print()
print("Transition tensor:\n", env.TransitionTensor())
print()
print("Observation tensor:\n", env.ObservationTensor())
print("\\end{OutputCode}")  #| hide_line

\begin{OutputCode}
SingleAgentUncertainDecisionProblem_1.0_0.5_0.5

State set: ['A', 'B']

Observation set: ['A', 'a', 'b', 'B']

Action set: ['A', 'B']

Transition tensor:
 [[[0.5 0.5]
  [0.5 0.5]]

 [[0.5 0.5]
  [0.5 0.5]]]

Observation tensor:
 [[[0.30853754 0.38292492 0.24173034 0.0668072 ]
  [0.0668072  0.24173034 0.38292492 0.30853754]]]
\end{OutputCode}


## Two-agent environment

In addition to the nosiy observations of the nature state, agents in the two-agent environment can also observe the actions of the other agent. Thus, the environment here is composed of the natural environment and the social environment. 

In each episode, agents choose their action subsequently. The first agent chooses an action only with the help of the noisy signal of the natural environment. 
Then, the second agent chooses an action with the help of the noisy signal of the natural environment and the observation of the first agent's action.
In each episode the order of agents is randomly chosen.

In [19]:
 #| export
class TwoAgentUncertainDecisionProblem(ebase):

    def __init__(self, 
                 noiselevel: float, # standard deviation of natural uncertainty
                 probabilityA: float = 0.5, # probability of nature to be in A
                 likelydistance: float = 1): # distance when agent's believe become *very* likely
        assert probabilityA >= 0 and probabilityA <= 1 

        self.env_A = probabilityA
        self.noise = noiselevel 
        self.likelydistance = likelydistance
        self.probs = probabilities(noiselevel, likelydistance)
        
        self.N = 2
        self.M = len(self.actions()[0])
        self.Z = len(self.states())
        self.Q = len(self.observations()[0])

        # inital states
        super().__init__()

The environmental states are composed of the natural states and the social states. The natural states are identical to the single-agent environment. 
The social states indicate the position of each agent and the action of the other agent,

```
'12.', '.1A', '.1B', '21.', '.2A', '.2B'
```

For example, the social state `12.` indicates that the first agent is behind the second agents which has not chosen any action yet. The social state `.1A` indicates that the second agent has chosen action *A*, and now, the first agent is about to choose an action relevant to obtain reward or punishment from the envrionment. In the social state `.1B`, the second agent has chosen action *B*. When it is not an agents turn to choose, their action simply has no effect and the mechanics of the enviroment.


In [20]:
 #| export
@patch
def states(self:TwoAgentUncertainDecisionProblem):
    nAorB = ["A", "B"]  # nature's state
    flow = ['12.', '.1A', '.1B', '21.', '.2A', '.2B']  # social state
    return [f'{n}{f}' for n, f in it.product(nAorB, flow)]

The agents' action set is identical to the single-agent environment.

In [21]:
 #| export
@patch
def actions(self:TwoAgentUncertainDecisionProblem):
   AorB = ['A', 'B']
   return [AorB, AorB]

The observation set in the two-agent environment is slighly more complicated. Essentially it is composed of the noisy signal of the natural environment  (in the first position) and the observation of the other agent's action (in the second position). Additionally, there is a dummy observation, `..`, for the agent which is currently not active.

In [22]:
 #| export
@patch
def observations(self:TwoAgentUncertainDecisionProblem):
    obs = ['A.', 'a.', 'b.', 'B.',  # for the first agent in line
           'AA', 'aA', 'bA', 'BA',  # for the second agent in line      
           'AB', 'aB', 'bB', 'BB', 
           '..']                    # for the inactive agent 
    return [obs, obs]

The transition tensor is slightly more complicated than in the single-agent environment. It models the movement and signaling of other agent's choices.

In [23]:
 #| export
@patch
def TransitionTensor(self:TwoAgentUncertainDecisionProblem):
    dim = np.concatenate(([self.Z], [self.M for _ in range(self.N)], [self.Z]))
    Tsas = np.ones(dim) * (-1)

    for index, _ in np.ndenumerate(Tsas):
        Tsas[index] = self._transition(index[0], index[1:-1], index[-1])
    return Tsas
    
@patch
def _transition(self:TwoAgentUncertainDecisionProblem, s, jA, s_):
    # translating states and actions into coordinates
    nat, pos2, pos1, choice = self.states()[s]          # current state  
    nat_, pos2_, pos1_, choice_ = self.states()[s_]     # next state

    # in movement state: 
    #   pos2= agent that plays second, pos1=agent that plays, choise='.'; 
    
    # in final state: 
    #   pos2='.', pos1=agent that plays, choice= choice of previous agent
    
    i = int(pos1)-1  # acting agent                       
    act = self.actions()[i][jA[i]] # action `A` or `B` of acting agent
    
    prop = 0.0  # default probability
    
    if choice != '.' and choice_ == '.':  # "final" states
        if nat_ == 'A':
            prop = self.env_A/2 
            # *1/2 because there are 2 possible next states with nat_='A'
        else:
            prop = (1-self.env_A)/2
                
    elif nat == nat_ and pos2 == pos1_:  # movement
        if act == choice_: prop = 1.0
            
    return prop

The reward tensor is very similar than in the single-agent environment. The only difference is that we scale the reward by a factor of 2 because also the episode length is doubled. In each episode, each agent has to wait for the other agent to make a choice. To have both environment with comparable average rewards, we scale the reward by a factor of 2.

In [24]:
 #| export
@patch
def RewardTensor(self:TwoAgentUncertainDecisionProblem):
    """Get the Reward Tensor R[i,s,a1,...,aN,s']."""
    dim = np.concatenate(([self.N], [self.Z], [self.M for _ in range(self.N)],
                          [self.Z]))
    Risas = np.zeros(dim)

    for index, _ in np.ndenumerate(Risas):
        Risas[index] = self._reward(index[0], index[1], index[2:-1], index[-1])
    return Risas

@patch
def _reward(self:TwoAgentUncertainDecisionProblem, i, s, jA, s_):
    # translating states and actions into coordinates
    nat, pos2, pos1, choice = self.states()[s]
    nat_, pos2_, pos1_, choice_ = self.states()[s_]
        
    j = int(pos1) - 1  #acting agent
    act = self.actions()[j][jA[j]]  # action `A` or `B` of acting agent           

    reward = 0  # default reward

    if j == i:  # if i is acting agent
        # if act is correct (according to nature)
        if act == nat: reward = 2
        elif act != nat: reward = -2
    return reward 

The observation tensor in the two-agent environment is more complicated than in the single-agent environment, as it combines the noisy observation from the nature state with the observation of the other agent's action.

In [25]:
 #| export
@patch
def ObservationTensor(self:TwoAgentUncertainDecisionProblem):
    """Returns Oiso. Probability the agent i makes observation o in state s"""
    Oiso = np.ones((self.N, self.Z, self.Q))
    
    for index, _ in np.ndenumerate(Oiso):
        Oiso[index] = self._observation(index[0], index[1], index[2])
    return Oiso

@patch
def _observation(self:TwoAgentUncertainDecisionProblem, i, s, o):
    # translating states and actions into coordinates       
    nat, _, pos1, choice = self.states()[s]
    nobs, sobs = self.observations()[i][o]  #nature obs., social obs.
              
    # observations probability composed of action obs and env obs
    pA, pE = 0.0, 0.0
        
    if nobs != ".":
        if choice == sobs:   #agent observes the true choice of predecessor
            pA = 1.0      
                
        if int(pos1)-1 == i:  # only pos1 agent observes something
            if nobs.upper() == nat and nobs.isupper():
                pE = self.probs[0]  #prop(A) if env in A, prop(B) if env in B  
            if nobs.upper() == nat and not nobs.isupper():
                pE = self.probs[1]  #prop(a) if env in A, prop(b) if env in B 
            if nobs.upper() != nat and not nobs.isupper():
                pE = self.probs[2]  #prop(b) if env in A, prop(a) if env in B
            if nobs.upper() != nat and nobs.isupper():
                pE = self.probs[3]  #prop(B) if env in A, prop(A) if env in B 
        
    elif nobs == "." and sobs == ".":
        #agent i is not acting agent -> makes no observation
        if int(pos1)-1 != i: pA = pE = 1.0
    
    return pE * pA

As above, the environment also gets an `id` method return a string identifier of the environment to be used in saving and retrieving results data.

In [26]:
 #| export
@patch
def id(self:TwoAgentUncertainDecisionProblem):
        """
        Returns id string of environment
        """
        id = f"{self.__class__.__name__}_"+\
            f"{self.noise}_{self.env_A}_{self.likelydistance}"
        return id
    

Executing some basic tests of the environment,

In [27]:
#| output: asis
env = TwoAgentUncertainDecisionProblem(noiselevel=1.0,
                                       probabilityA=0.6,
                                       likelydistance=0.5)

print("\\begin{OutputCode}") #| hide_line
print(env.id())
print()
print("State set:", env.Sset)
print()
print("Observation set:", env.Oset[0])
print()
print("Action set:", env.Aset[0])
print()
print("Transition tensor's shape:\n", env.TransitionTensor().shape)
print()
print("Observation tensor's shape:\n", env.ObservationTensor().shape)
print("\\end{OutputCode}") #| hide_line

\begin{OutputCode}
TwoAgentUncertainDecisionProblem_1.0_0.6_0.5

State set: ['A12.', 'A.1A', 'A.1B', 'A21.', 'A.2A', 'A.2B', 'B12.', 'B.1A', 'B.1B', 'B21.', 'B.2A', 'B.2B']

Observation set: ['A.', 'a.', 'b.', 'B.', 'AA', 'aA', 'bA', 'BA', 'AB', 'aB', 'bB', 'BB', '..']

Action set: ['A', 'B']

Transition tensor's shape:
 (12, 2, 2, 12)

Observation tensor's shape:
 (2, 12, 13)
\end{OutputCode}


Finally, we export the environment classes into their own module,

In [28]:
import nbdev
nbdev.export.nb_export("j01_ENVI_UncertainDecisionProblem.ipynb", "_code")